# 第 01 章：环境配置与智能体底层逻辑 (实战版)

> **学习定位**：本章属于「第一阶段：把 LangChain 当开发入口」。
> 这一章的 notebook 目标很明确：直接把 `invoke`、`bind_tools`、`|`、`create_agent`、`astream` 这几种入口跑起来，建立最初的手感。

本章实验会按这个顺序推进：
1. 初始化模型并验证连通性
2. 体验一次最小 `invoke()` 调用
3. 用 `bind_tools()` 给模型绑定工具
4. 体验一次 `|` Runnable 管道
5. 组织 Agent 的标准化输入
6. 运行带工具的 `create_agent()`
7. 正确消费 `astream()` 的流式输出


## 2. 环境诊断与模型初始化

先确认网络和 API 能力，然后用 `init_chat_model` 建立统一模型入口。

In [8]:
import requests

try:
    r = requests.get("https://api.deepseek.com", timeout=5)
    print(f"DeepSeek API 连通性测试成功：{r.status_code}")
except Exception as e:
    print(f"DeepSeek API 连通性测试失败: {e}")
    print("提示：如果连接超时或 SSL 错误，请检查网络代理、VPN 或公司网络策略。")


DeepSeek API 连通性测试成功：401


In [9]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    streaming=True,
)

print(f"模型加载成功：{llm.__class__.__name__}")


模型加载成功：ChatDeepSeek


## 3. 第一种入口：直接使用 `invoke()`

当你只想做一次最小问答验证时，直接调用 `llm.invoke(...)` 就够了。

In [10]:
reply = llm.invoke("请用一句话介绍 LangChain。")
print(reply.content)

LangChain是一个用于开发由大型语言模型驱动的应用程序的框架，它通过模块化组件和链式调用简化了与LLM集成、数据连接和复杂工作流构建的过程。


## 4. 给模型绑定工具：`bind_tools()`

`bind_tools()` 仍然属于“增强 LLM”这一层。它会让模型具备生成 tool call 的能力，但不会像 `create_agent()` 那样自动帮你跑完整工具循环。


In [11]:
from langchain.tools import tool

@tool
def get_system_time(query: str) -> str:
    """返回当前系统的具体时间。当用户询问时间相关的实时信息时，必须使用此工具。"""
    import datetime
    return f"北京时间：{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

llm_with_tools = llm.bind_tools([get_system_time])
ai_msg = llm_with_tools.invoke("现在北京几点了？")
print(ai_msg.tool_calls)


[{'name': 'get_system_time', 'args': {'query': '北京当前时间'}, 'id': 'call_00_UBmai7rbxHd8NyfadOWsDokg', 'type': 'tool_call'}]


## 5. 第二种入口：使用 `|` 组合固定流程

当流程是确定的，比如“Prompt -> LLM -> Parser”，可以使用 Runnable 管道。这里既可以接普通 `llm`，也可以接已经 `bind_tools()` 过的模型。


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个简洁、准确的技术助理。"),
    ("user", "{question}"),
])

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"question": "什么是 Runnable？"}))

llm_with_tools_chain = prompt | llm_with_tools
ai_msg_from_chain = llm_with_tools_chain.invoke({"question": "告诉我北京几点了？"})
print(ai_msg_from_chain.tool_calls)


**Runnable** 是 Java 中的一个核心接口，用于定义可在线程中执行的任务。它位于 `java.lang` 包中，是 Java 并发编程的基础之一。

### 主要特点：
1. **单一抽象方法**：只包含一个方法 `void run()`，用于定义任务逻辑。
2. **无返回值**：`run()` 方法不返回结果，也不抛出受检异常。
3. **线程兼容**：通常与 `Thread` 类或线程池（如 `ExecutorService`）结合使用。

### 基本用法：
```java
// 1. 实现 Runnable 接口
class MyTask implements Runnable {
    @Override
    public void run() {
        System.out.println("任务执行中");
    }
}

// 2. 使用 Thread 启动任务
Thread thread = new Thread(new MyTask());
thread.start();

// 3. 使用 Lambda 表达式简化（Java 8+）
Runnable task = () -> System.out.println("Lambda 任务");
new Thread(task).start();
```

### 与 Callable 的区别：
| 特性       | Runnable                     | Callable                     |
|------------|------------------------------|------------------------------|
| 返回值     | 无 (`void`)                  | 有（泛型类型）               |
| 异常       | 不能抛出受检异常             | 可以抛出受检异常             |
| 使用场景   | 简单任务、无需返回结果       | 需要结果或异常处理的复杂任务 |

### 实际应用：
- **线程池任务提交**：通过 `ExecutorService.execute(runnable)` 执行。

## 6. 消息组织：给 Agent 准备标准化输入

进入 Agent 之前，我们先把输入组织方式固定下来。最常见的是三层结构：System -> History -> User。

In [13]:
def prepare_inputs(user_query: str, chat_history: list | None = None):
    history = chat_history or []
    return {
        "messages": [
            ("system", "你是一个实事求是的助手。如果不知道时间，请使用工具。"),
            *history,
            ("user", user_query),
        ]
    }

example_input = prepare_inputs("你好，你是谁？")
print(example_input)

{'messages': [('system', '你是一个实事求是的助手。如果不知道时间，请使用工具。'), ('user', '你好，你是谁？')]}


## 7. 第三种入口：`create_agent()`

当你希望模型自己决定是否调用工具，并在工具执行后继续完成任务时，就该进入 Agent 路线。

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_system_time],
)

result = agent.invoke({"messages": [("user", "告诉我北京几点了？")]})
print(result["messages"][-1].content)


现在是北京时间：2026年4月15日 15:35:24（下午3点35分24秒）。


## 8. 正确消费 `astream()`

这里我们显式使用 `version="v2"`。在这个版本下，`agent.astream(..., stream_mode="messages")` 的每个流项会先被包装成事件字典，所以应该先读取 `event`，再从 `event["data"]` 中取出 `(chunk, metadata)`。


In [15]:
async def run_streaming_demo(query: str):
    input_dict = prepare_inputs(query)
    full_response = None

    print(f"\n--- 提问: {query} ---\n")

    async for event in agent.astream(
        input_dict,
        stream_mode="messages",
        version="v2",
    ):
        if event["type"] != "messages":
            continue

        chunk, metadata = event["data"]

        if metadata.get("langgraph_node") == "model" and chunk.content:
            print(chunk.content, end="", flush=True)

        if metadata.get("langgraph_node") == "model":
            full_response = chunk if full_response is None else full_response + chunk

    return full_response

final_msg = await run_streaming_demo("告诉我北京几点了？")
print(f"\n\n[聚合完毕] 消费 Token 总计: {final_msg.usage_metadata['total_tokens']}")



--- 提问: 告诉我北京几点了？ ---

我来帮您查询北京当前的时间。现在是北京时间：2026年4月15日 15:35:29

[聚合完毕] 消费 Token 总计: 819


## 9. 小结

到这里你应该已经能区分三件事：

1. `invoke()`：单次调用，适合最小验证。
2. `|`：固定顺序的 Runnable 编排。
3. `create_agent()`：带工具、带决策循环的 Agent 入口。

后面的章节会继续基于这三种入口展开，但你现在已经有了第一张不会迷路的地图。